# Signature verification - 256x512 training and evaluation (A/B/C)

Trains the Stage 3 ConvNeXt-Tiny metric-learning encoder and evaluates it on a Kaggle T4.

**Canonical canvas:** `256 x 512` (H x W), grayscale, white background. This notebook
does not use the legacy 384x192 upstream.

**Dataset prerequisite.** Upload `training_clean_256x512.zip` as a private Kaggle
dataset. The archive is flat: `manifest_aug.csv` at the root, with `BHSig260/`, `CEDAR/`
and `SigResize/` beside it. The manifest must have 20,548 rows (14,401 train / 3,114 val
/ 3,033 test) and 421 signers.

**Notebook settings**

* Accelerator: **GPU T4**
* Internet: **ON** (the pretrained ConvNeXt weights are downloaded on the first run)
* Every output goes to `/kaggle/working`; the attached dataset stays read-only.

**Runs.** A, B and C share the seed, epochs, sampler and test split. The only change is
the training data or augmentation:

| run | train data | augmentation |
| --- | --- | --- |
| A | BHSig260 + CEDAR + SigResize | off (`--no-augment`) |
| B | BHSig260 + CEDAR + SigResize | on (`--augment`) |
| C | BHSig260 + CEDAR (no SigResize) | on (`--augment`) |

Augmentation touches the train split only; validation and test pixels are never
augmented. Checkpoints are selected by validation skilled-forgery EER with 5 references,
the same protocol and reference count that `training.evaluate` reports.

In [ ]:
# Cell 1 - clone this repository and install only what Kaggle is missing.
REPO_URL = "https://github.com/yogeshY0/signautre-256-512.git"
REPO_DIR = "/kaggle/working/repo"

!rm -rf {REPO_DIR}
!git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -1

# Kaggle already ships torch/torchvision; reinstalling them wastes ~2 GB.
!pip install -q --no-input timm opencv-python scikit-learn

In [ ]:
# Cell 2 - environment check. Turn Internet ON before running (pretrained weights).
import platform
import sys

import cv2
import timm
import torch

print("python :", sys.version.split()[0], platform.platform())
print("torch  :", torch.__version__, "| cuda:", torch.cuda.is_available())
print("gpu    :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("timm   :", timm.__version__, "| cv2:", cv2.__version__)
assert torch.cuda.is_available(), "Select the T4 GPU accelerator before running"

In [ ]:
# Cell 3 - locate the attached dataset. It is mounted read-only under /kaggle/input.
import glob
from pathlib import Path

candidates = sorted(glob.glob("/kaggle/input/**/manifest_aug.csv", recursive=True))
assert candidates, "no manifest_aug.csv under /kaggle/input - attach the 256x512 dataset"
MANIFEST = Path(candidates[0])
IMAGE_ROOT = MANIFEST.parent
print("manifest  :", MANIFEST)
print("image root:", IMAGE_ROOT)
print("top level :", sorted(path.name for path in IMAGE_ROOT.iterdir()))

In [ ]:
# Cell 4 - counts and path resolution. Fails loudly before anything is trained.
import collections
import csv

rows = list(csv.DictReader(MANIFEST.open(encoding="utf-8")))
print("rows    :", len(rows))
print("splits  :", dict(collections.Counter(row["split"] for row in rows)))
print("signers :", len({row["signer_id"] for row in rows}))
print("datasets:", dict(collections.Counter(row["dataset"] for row in rows)))
missing = [
    row["image_path"] for row in rows if not (IMAGE_ROOT / row["image_path"]).is_file()
]
print("missing :", len(missing), missing[:3])
assert len(rows) == 20548, f"expected 20548 rows, got {len(rows)}"
assert not missing, "manifest paths do not resolve under IMAGE_ROOT"
print("PASS: manifest and image paths")

In [ ]:
# Cell 5 - the data gate. Training must not start unless this prints PASS.
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "training.verify_data",
        "--manifest",
        str(MANIFEST),
        "--image-root",
        str(IMAGE_ROOT),
    ],
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr, file=sys.stderr)
assert result.returncode == 0 and "RESULT: PASS" in result.stdout, (
    "verify_data did not pass; do not train on this dataset"
)
print("PASS: verify_data (3/3 sections)")

## 1. Train A, B and C

`--phase1-epochs 5 --phase2-epochs 40` is 5 warmup epochs with a frozen backbone and
semi-hard mining, then 40 fine-tune epochs on the last two ConvNeXt stages with
batch-hard mining and cosine LR. `--select-by eer` keeps the checkpoint with the lowest
validation skilled-forgery EER. `--balance-datasets` is deliberately off: the data audit
did not show that equal corpus shares help, so it stays an opt-in ablation flag. Every
epoch's JSON line contains `epoch_seconds` and `elapsed_seconds`.

In [ ]:
# Cell 6 - shared settings. A, B and C differ only in the flags on their own cell.
SEED = 42
COMMON = (
    f"--manifest {MANIFEST} --image-root {IMAGE_ROOT} --already-preprocessed "
    f"--image_height 256 --image_width 512 --device cuda --workers 2 --seed {SEED} "
    "--phase1-epochs 5 --phase2-epochs 40 --select-by eer"
)
print(COMMON)

In [ ]:
# Cell 7 - run A: all corpora, no augmentation (the baseline).
!python -u -m training.train_siamese {COMMON} --no-augment \
  --output-dir /kaggle/working/ckpt_A_noaug 2>&1 | tee /kaggle/working/train_A.log

In [ ]:
# Cell 8 - run B: all corpora, augmentation on (the recommended run).
!python -u -m training.train_siamese {COMMON} --augment \
  --output-dir /kaggle/working/ckpt_B_aug 2>&1 | tee /kaggle/working/train_B.log

In [ ]:
# Cell 9 - run C: drop SigResize, keep augmentation. This isolates what SigResize adds.
!python -u -m training.train_siamese {COMMON} --train-datasets BHSig260,CEDAR --augment \
  --output-dir /kaggle/working/ckpt_C_no_sigresize 2>&1 | tee /kaggle/working/train_C.log

## 2. Evaluate A, B and C on the same test split

These are the deep baselines: no fusion config is passed. The threshold is chosen on
validation signers and applied unchanged to test, so test is never tuned on.
`--target-far 0.001` adds the fixed-FAR operating point, and `per_dataset` reports every
corpus separately. The comparison table below is the one to read: a run only counts as
better if it improves the per-dataset numbers, not only the pooled average.

In [ ]:
# Cell 10 - evaluate every checkpoint on the identical test split.
import json
import subprocess

Path("/kaggle/working/reports").mkdir(parents=True, exist_ok=True)
CHECKPOINTS = {
    "A_noaug": "/kaggle/working/ckpt_A_noaug/best_model.pt",
    "B_aug": "/kaggle/working/ckpt_B_aug/best_model.pt",
    "C_no_sigresize": "/kaggle/working/ckpt_C_no_sigresize/best_model.pt",
}
REPORTS = {}
for name, checkpoint in CHECKPOINTS.items():
    report_path = f"/kaggle/working/reports/eval_{name}.json"
    print(f"\n=== evaluate {name} ===", flush=True)
    subprocess.run(
        [
            sys.executable,
            "-m",
            "training.evaluate",
            "--checkpoint",
            checkpoint,
            "--manifest",
            str(MANIFEST),
            "--image-root",
            str(IMAGE_ROOT),
            "--already-preprocessed",
            "--image_height",
            "256",
            "--image_width",
            "512",
            "--references",
            "5",
            "--device",
            "cuda",
            "--batch-size",
            "32",
            "--workers",
            "2",
            "--target-far",
            "0.001",
            "--fusion-report",
            report_path,
        ],
        check=True,
    )
    REPORTS[name] = json.loads(Path(report_path).read_text())
print("reports:", sorted(REPORTS))

In [ ]:
# Cell 11 - pooled and per-dataset comparison of A, B and C.
import pandas as pd

table = []
for name, report in REPORTS.items():
    row = {
        "run": name,
        "val_skilled": report["validation_eer_skilled"],
        "test_skilled": report["test"]["eer_skilled"],
        "test_random": report["test"]["eer_random"],
    }
    for dataset, metrics in report["per_dataset"]["test"].items():
        row[f"test_{dataset}"] = metrics["eer_skilled"]
    table.append(row)
comparison = pd.DataFrame(table).set_index("run").sort_index()
print(comparison.round(4).to_string())

## 3. Fusion (report-only, opt-in)

Fusion is fitted on the validation split with **skilled-forgery negatives only**
(`--max-random-queries 0`), then evaluated once on test. The report's `deep_vs_fused`
block prints the per-dataset skilled EER and recommends `deep_only` unless fusion
improves **every** corpus on validation. Do not ship a fused scorer that wins only on the
pooled average. The `calibration` block compares one global threshold with a threshold
fitted on each dataset's own validation split, so a miscalibrated corpus is visible.

In [ ]:
# Cell 12 - fit fusion on B's validation split (skilled negatives only).
!python -u -m training.fit_fusion \
  --checkpoint /kaggle/working/ckpt_B_aug/best_model.pt \
  --manifest {MANIFEST} --image-root {IMAGE_ROOT} --already-preprocessed \
  --image_height 256 --image_width 512 --references 5 \
  --device cuda --batch-size 32 --workers 2 \
  --max-random-queries 0 \
  --output /kaggle/working/fusion_B.json

In [ ]:
# Cell 13 - evaluate B deep vs fused, with the calibration table.
fusion_report_path = "/kaggle/working/reports/eval_B_fused.json"
!python -u -m training.evaluate \
  --checkpoint /kaggle/working/ckpt_B_aug/best_model.pt \
  --manifest {MANIFEST} --image-root {IMAGE_ROOT} --already-preprocessed \
  --image_height 256 --image_width 512 --references 5 \
  --device cuda --batch-size 32 --workers 2 \
  --max-random-queries 0 --target-far 0.001 \
  --fusion-config /kaggle/working/fusion_B.json \
  --fusion-report {fusion_report_path}

import json

fused_report = json.loads(Path(fusion_report_path).read_text())
verdict = fused_report["deep_vs_fused"]
print("FUSION VERDICT:", verdict["recommendation"],
      "| helps every dataset:", verdict["helps_every_dataset"])
print("global vs per-dataset calibration (deep):")
for dataset, entry in fused_report["calibration"]["deep"]["datasets"].items():
    global_rates = entry["test_at_global_threshold"]
    dataset_rates = entry["test_at_dataset_threshold"]
    print(
        f"  {dataset:<12} thr_ds={entry['threshold']:.4f} "
        f"FRR_global={global_rates['frr_at_threshold']:.4f} "
        f"FRR_ds={dataset_rates['frr_at_threshold']:.4f}"
    )

## 4. Per-corpus embedding audit

Same enrolment protocol as evaluation, with per-dataset EER, FAR/FRR at two fixed
budgets, score distributions and histograms. This is where SigResize shows up if it is
the weak corpus; the audit is read-only.

In [ ]:
# Cell 14 - per-dataset audit of B (the recommended checkpoint).
!python -u -m training.audit_embeddings \
  --checkpoint /kaggle/working/ckpt_B_aug/best_model.pt \
  --manifest {MANIFEST} --image-root {IMAGE_ROOT} --already-preprocessed \
  --image_height 256 --image_width 512 --references 5 \
  --device cuda --batch-size 32 --workers 2 \
  --max-random-queries 2000 --target-fars 0.001,0.01 \
  --hist-dir /kaggle/working/audit_histograms

## 5. Package the artifacts

`/kaggle/working` is what Kaggle offers for download when the session ends.

In [ ]:
# Cell 15 - tar the checkpoints, reports and logs, then list the archive.
!mkdir -p /kaggle/working/reports
!tar -czf /kaggle/working/artifacts_256x512.tar.gz -C /kaggle/working \
  ckpt_A_noaug/best_model.pt ckpt_B_aug/best_model.pt ckpt_C_no_sigresize/best_model.pt \
  fusion_B.json train_A.log train_B.log train_C.log reports
!ls -lh /kaggle/working/artifacts_256x512.tar.gz /kaggle/working/ckpt_*/best_model.pt

## 6. Build the Kaggle dataset zip (run locally, not on Kaggle)

This flattens `training_clean/`: the manifests stay at the archive root and the three
corpus folders sit beside them. `.DS_Store`, `aug_preview.png`, `preview.py` and any old
zips are excluded.

```bash
cd /Users/yogesh/c0DE/Signature-Verification-FIRST-/training_clean
zip -r training_clean_256x512.zip . \
  -x '.DS_Store' -x '*/.DS_Store' -x 'aug_preview.png' -x 'preview.py' -x '*.zip'
ls -lh training_clean_256x512.zip
du -h training_clean_256x512.zip
```

Upload the resulting zip as a private Kaggle dataset and attach it to this notebook.

In [ ]:
# Cell 16 - if the zip is present in the session, print its size; otherwise print the
# local build command above. The zip is normally uploaded from the local machine.
from pathlib import Path

zip_path = Path("/kaggle/working/training_clean_256x512.zip")
if zip_path.exists():
    print(f"{zip_path}: {zip_path.stat().st_size / 1e6:.1f} MB")
else:
    print("training_clean_256x512.zip not present in the session; build it locally with the command above.")

## Reading the results

* Report **`per_dataset.test.<corpus>.eer_skilled`**, not only the pooled average. A run
  is only better if the per-corpus numbers improve too.
* `deep_vs_fused.recommendation` must be `keep_fusion` before a fused scorer is shipped.
* `calibration` shows whether one global threshold is miscalibrated for a corpus; the
  per-dataset threshold is fitted on that corpus's validation signers and applied
  unchanged to test.
* The curated numbers come from a real trained checkpoint. Do not quote metrics from a
  1-epoch, random-init smoke run.
* `/kaggle/working` is wiped when the session ends: download
  `artifacts_256x512.tar.gz` and `fusion_B.json` before stopping.

**Canonical canvas:** 256 x 512. Do not merge the old 384 x 192 upstream into this branch.